In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

from bio_network.engine.neurons import IzhikevichPopulation
from bio_network.engine.scheduler import simulate
from bio_network.engine.synapses import RandomSynapses
from bio_network.engine.synapses_sparse import SparseSynapses
from bio_network.viz.raster import plot_population_rate, plot_raster

N_EXC, N_INH = 800, 200
T_MS = 1000
SEED = 42

OUTPUT_DIR = os.path.join(os.getcwd(), "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output directory:", OUTPUT_DIR)


In [ ]:
# Run the dense (golden reference) and sparse (event-driven) engines.

dense_pop = IzhikevichPopulation(seed=SEED)
dense_rec = simulate(dense_pop, T_ms=T_MS, seed=SEED, engine="dense")

# The sparse engine uses an equal per-neuron synaptic budget as dense so the
# two can be compared statistically. Chaotic trajectories are expected to
# diverge after ~100 ms; equivalence is statistical, not spike-exact.
sparse_syn = SparseSynapses(n_excit=N_EXC, n_inhib=N_INH, out_degree=N_EXC + N_INH, seed=SEED)
sparse_pop = IzhikevichPopulation(seed=SEED)
sparse_rec = simulate(sparse_pop, sparse_syn, T_ms=T_MS, seed=SEED, engine="sparse")

print(f"dense : {dense_rec.times_ms.size} spikes")
print(f"sparse: {sparse_rec.times_ms.size} spikes")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
plot_raster(dense_rec, ax=axes[0][0])
axes[0][0].set_title("Dense (golden reference)")
plot_raster(sparse_rec, ax=axes[0][1])
axes[0][1].set_title("Sparse (event-driven, delays)")
axes[0][1].set_ylabel("Neuron index")

plot_population_rate(dense_rec, bin_ms=10, ax=axes[1][0])
axes[1][0].set_ylabel("Population rate (Hz)")
axes[1][0].set_xlabel("Time (ms)")
plot_population_rate(sparse_rec, bin_ms=10, ax=axes[1][1])
axes[1][1].set_xlabel("Time (ms)")

fig.suptitle("Dense vs sparse: spike rasters (top) and population rate (bottom, 10 ms bins)")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "m1_dense_vs_sparse_rasters.png"), dpi=150)
plt.show()


In [ ]:
def summarize(rec):
    rates = rec.mean_rates_hz()
    return (rec.times_ms.size, rates[:N_EXC].mean(), rates[N_EXC:].mean(), rates.max())

print(f"{'engine':<8} {'total':>7} {'mean exc':>9} {'mean inh':>9} {'max':>6}")
for name, rec in (("dense", dense_rec), ("sparse", sparse_rec)):
    total, exc, inh, mx = summarize(rec)
    print(f"{name:<8} {total:7d} {exc:9.2f} {inh:9.2f} {mx:6.2f}")


In [ ]:
import time
import tracemalloc

def bench(engine, size):
    n_exc = int(size * 0.8)
    n_inh = size - n_exc
    if engine == "dense":
        s = RandomSynapses(n_pre_excit=n_exc, n_pre_inhib=n_inh, seed=SEED)
        s.S = s.S.astype(np.float32)
    else:
        s = SparseSynapses(n_excit=n_exc, n_inhib=n_inh, out_degree=100, seed=SEED)
    pop = IzhikevichPopulation(n_excitatory=n_exc, n_inhibitory=n_inh)
    tracemalloc.start()
    t0 = time.perf_counter()
    try:
        rec = simulate(pop, s, T_ms=200, seed=SEED, engine=engine)
        wall = time.perf_counter() - t0
        mb = tracemalloc.get_traced_memory()[1] / 1e6
        row = f"{size:6d}  {engine:<6}  {wall:7.2f}s  {mb:9.1f}MB  "
        row += f"{rec.mean_rates_hz()[:n_exc].mean():5.2f}Hz"
    except MemoryError:
        row = f"{size:6d}  {engine:<6}  infeasible (>RAM)"
    finally:
        tracemalloc.stop()
    return row

print(f"{'N':>6}  {'engine':<6}  {'time':>9}  {'RAM':>12}  {'mean exc':>9}")
for size in (1000, 10000):
    for eng in ("dense", "sparse"):
        print(bench(eng, size))
